# Golden Cross 백테스트

SMA(50) > SMA(200) 골든크로스 전략 — Phase 1 완료 기준 검증 노트북.

**전제**: `scripts/backfill_us_daily.py`로 스냅샷이 생성되어 있어야 합니다.
```bash
uv run python scripts/backfill_us_daily.py --symbols AAPL MSFT SPY --start 2015-01-01
```

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from ginlix_data_sdk import parquet_store as store
from ginlix_backtest import indicators as ind
from ginlix_backtest.engine import portfolio
from ginlix_backtest.io import persist

SNAPSHOT = 'us-2026-04-23'   # backfill 후 실제 스냅샷 ID로 변경
SYMBOL   = 'AAPL'

print('사용 가능한 스냅샷:', store.list_snapshots())

## 1. 데이터 로드

In [ ]:
close  = store.load_prices([SYMBOL], snapshot_id=SNAPSHOT)[SYMBOL]
spy    = store.read_benchmark('SPY', snapshot_id=SNAPSHOT)['split_adj_close']

print(f'{SYMBOL}: {len(close)} bars  |  {close.index[0].date()} → {close.index[-1].date()}')
close.tail()

## 2. 시그널 생성

In [ ]:
fast = ind.sma(close, 50)
slow = ind.sma(close, 200)

# 골든크로스: 50일선이 200일선을 위로 돌파
entries = (fast > slow) & (fast.shift(1) <= slow.shift(1))
# 데드크로스: 50일선이 200일선을 아래로 돌파
exits   = (fast < slow) & (fast.shift(1) >= slow.shift(1))

print(f'매수 시그널: {entries.sum()}건  |  매도 시그널: {exits.sum()}건')

## 3. 백테스트 실행

`from_signals`이 내부적으로:
- 시그널을 1바 뒤로 shift (next-bar fill, look-ahead 방지)
- US_DEFAULT 수수료(SEC/TAF) + 3bps 슬리피지 자동 적용
- SPY 대비 벤치마크 지표 자동 계산

In [ ]:
result = portfolio.from_signals(
    close=close,
    entries=entries,
    exits=exits,
    symbol=SYMBOL,
    benchmark_close=spy,
    initial_capital=100_000,
)

print(result)

## 4. 성과 지표

In [ ]:
import pandas as pd

m = result.metrics
bm = result.benchmark_metrics

pd.DataFrame({
    '전략': {
        'CAGR':          f"{m['cagr']:.1%}",
        'Sharpe':        f"{m['sharpe']:.2f}",
        'Sortino':       f"{m['sortino']:.2f}",
        'Max Drawdown':  f"{m['max_dd']:.1%}",
        'Win Rate':      f"{m.get('win_rate', float('nan')):.1%}",
        '거래 횟수':      m['n_trades'],
    },
    'vs SPY': {
        'SPY CAGR (참고)': f"{bm.get('benchmark_total_return', float('nan')):.1%}" if bm else '-',
        '초과 수익':       f"{bm.get('excess_return', float('nan')):.1%}" if bm else '-',
        'Information Ratio': f"{bm.get('information_ratio', float('nan')):.2f}" if bm else '-',
        'Beta':           f"{bm.get('beta', float('nan')):.2f}" if bm else '-',
        '상관계수':        f"{bm.get('correlation', float('nan')):.2f}" if bm else '-',
        '': '',
    }
})

## 5. 에쿼티 커브

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# 에쿼티 커브
equity = result.equity
spy_norm = spy.reindex(equity.index, method='ffill') / spy.reindex(equity.index, method='ffill').iloc[0] * 100_000

axes[0].plot(equity, label=f'Golden Cross ({SYMBOL})', linewidth=1.5)
axes[0].plot(spy_norm, label='SPY (B&H)', linewidth=1.5, alpha=0.7, linestyle='--')
axes[0].set_ylabel('Portfolio Value ($)')
axes[0].legend()
axes[0].set_title(f'Golden Cross SMA(50/200) — {SYMBOL}')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# SMA 오버레이
axes[1].plot(close, label='Price', linewidth=1, alpha=0.6)
axes[1].plot(fast, label='SMA 50', linewidth=1.2)
axes[1].plot(slow, label='SMA 200', linewidth=1.2)
buy_dates  = close[entries].index
sell_dates = close[exits].index
axes[1].scatter(buy_dates,  close[entries], marker='^', color='green', s=80, zorder=5, label='Buy')
axes[1].scatter(sell_dates, close[exits],   marker='v', color='red',   s=80, zorder=5, label='Sell')
axes[1].set_ylabel('Price ($)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. DB 저장 (선택)

DATABASE_URL 환경변수가 설정된 경우에만 실행됩니다.

In [ ]:
import os

STRATEGY_CODE = f"""
fast = ind.sma(close, 50)
slow = ind.sma(close, 200)
entries = (fast > slow) & (fast.shift(1) <= slow.shift(1))
exits   = (fast < slow) & (fast.shift(1) >= slow.shift(1))
"""

if os.environ.get('DATABASE_URL'):
    job_id = persist.save_result(
        result=result,
        strategy_name=f'Golden Cross SMA(50/200) {SYMBOL}',
        strategy_code=STRATEGY_CODE,
        snapshot_id=SNAPSHOT,
        params={'symbol': SYMBOL, 'fast': 50, 'slow': 200},
        tags=['golden_cross', 'trend_following'],
    )
    print(f'저장 완료. job_id = {job_id}')

    # 재현 검증
    saved = persist.load_run(job_id)
    print('저장된 Sharpe:', saved[0]['metrics']['sharpe'])
else:
    print('DATABASE_URL 미설정 — DB 저장 건너뜀')

## 7. vectorbt 풀 통계 (선택)

In [ ]:
result.stats